# Stakeholder Gym â€” Local-model evaluation (no API)

Runs the eval harness with open-weight models loaded via `transformers` on Colab's GPU. **Zero API keys, zero rate limits.**

Works on free-tier T4 (16GB VRAM) for any 4-bit model up to ~8B. Larger models (13B+) benefit from Colab Pro A100.

## 1. Setup

Replace `REPO_URL` with your actual repo URL once you push. If running locally, set `REPO_URL=None` and `REPO_DIR` to the repo path.

In [ ]:
!pip install -q transformers accelerate bitsandbytes pydantic networkx pyyaml

import os, subprocess

# >>> EDIT THIS: point at your own fork once you push to GitHub/HF.
REPO_URL = os.environ.get('STAKEHOLDER_GYM_REPO', 'https://github.com/SAISriram19/meta.git')
REPO_DIR = '/content/stakeholder-gym'

if not os.path.isdir(REPO_DIR):
    if REPO_URL and REPO_URL.startswith('http'):
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    else:
        raise RuntimeError('set REPO_URL or upload the repo folder to /content/stakeholder-gym')
%cd {REPO_DIR}
!ls

## 2. HuggingFace token (needed for gated models)

Llama-3.x, Gemma, Mistral-Instruct etc. are gated behind a free HF license. Get a token at https://huggingface.co/settings/tokens. Public models (Qwen, DeepSeek, etc.) work without one.

In [ ]:
from getpass import getpass
import os

if not os.environ.get('HF_TOKEN') and not os.environ.get('HUGGING_FACE_HUB_TOKEN'):
    tok = getpass('Paste your HF token (blank = public models only): ')
    if tok.strip():
        os.environ['HF_TOKEN'] = tok.strip()
        os.environ['HUGGING_FACE_HUB_TOKEN'] = tok.strip()
        from huggingface_hub import login
        login(token=tok.strip(), add_to_git_credential=False)
        print('HF token set.')
    else:
        print('No token â€” limiting to public models (Qwen, DeepSeek, etc).')

## 3. Pick models

All these fit on free T4 with 4-bit quantization. Uncomment to add; leave commented to skip.

**Gated (need HF token):**
- `meta-llama/Llama-3.1-8B-Instruct` â€” strong baseline, 4-bit ~5GB VRAM
- `meta-llama/Llama-3.2-3B-Instruct` â€” small + capable
- `google/gemma-2-9b-it` â€” tight on T4 but works in 4-bit
- `mistralai/Mistral-7B-Instruct-v0.3`

**Public (no token needed):**
- `Qwen/Qwen2.5-0.5B-Instruct` â€” tiny, fast sanity check
- `Qwen/Qwen2.5-1.5B-Instruct`
- `Qwen/Qwen2.5-3B-Instruct`
- `Qwen/Qwen2.5-7B-Instruct`
- `microsoft/Phi-3.5-mini-instruct`

In [ ]:
# --- CALIBRATED FOR FREE-TIER T4 (~10-15 min total runtime) ---
# Qwen 1.5B: too small to produce valid JSON reliably (0 pushbacks in tests).
# Qwen 3B+: starts producing usable actions. Phi-3.5 is the sweet spot for
# fast + capable. Uncomment 8B+ models once you're on Colab Pro or an A100.
MODELS = [
    'Qwen/Qwen2.5-3B-Instruct',        # public, fast, produces valid JSON
    'microsoft/Phi-3.5-mini-instruct',  # public, strong instruction-following
    'Qwen/Qwen2.5-7B-Instruct',        # public, best zero-shot on this env
    # Gated (needs HF token + license-grant on each model page):
    # 'meta-llama/Llama-3.1-8B-Instruct',
    # 'meta-llama/Llama-3.2-3B-Instruct',
    # 'google/gemma-2-9b-it',
    # 'mistralai/Mistral-7B-Instruct-v0.3',
]

# Start with L0 (30 steps) to verify pipeline, add L2 (120 steps) once you're confident.
SCENARIOS = ['L0_launch', 'L2_strategic_shift']
# 1 seed = ~7 min per model-pair. 3 seeds = ~20 min. Start with 1.
SEEDS = [0]
OUT_DIR = 'eval_outputs/hf_colab'

## 4. Run eval on each model sequentially

Models are loaded one at a time to avoid VRAM blowup. Each is freed before the next loads.

In [ ]:
import sys, time, gc, torch, json
sys.path.insert(0, '.')
from pathlib import Path
from eval.harness import EvalConfig, run_eval
from eval.policies import build_policy, _HF_CACHE

all_records_path = Path(OUT_DIR) / 'rollouts.jsonl'
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
total_t0 = time.time()

for m in MODELS:
    sub_out = Path(OUT_DIR) / m.replace('/', '__')
    print(f'\n=== {m} ===')
    try:
        policies = {f'hf:{m}': build_policy(f'hf:{m}')}
        config = EvalConfig(
            policies=policies,
            scenarios=SCENARIOS,
            seeds=SEEDS,
            out_dir=sub_out,
        )
        run_eval(config, verbose=True)
        # free the model before next iteration
        for key in list(_HF_CACHE.keys()):
            model, tok = _HF_CACHE[key]
            del model
            del tok
            del _HF_CACHE[key]
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as e:
        print(f'  FAILED: {e}')

print(f'\ntotal elapsed: {time.time()-total_t0:.1f}s')

## 5. Rule-based baselines (no GPU)

Same scenarios, same seeds, zero model loading. Gives you the reference curve.

In [ ]:
baseline_policies = {
    'sycophant': build_policy('sycophant'),
    'contrarian': build_policy('contrarian'),
    'keyword_principled': build_policy('keyword_principled'),
    'memory_aware': build_policy('memory_aware'),
}
baseline_config = EvalConfig(
    policies=baseline_policies,
    scenarios=SCENARIOS,
    seeds=SEEDS,
    out_dir=Path('eval_outputs/hf_colab_baselines'),
)
run_eval(baseline_config, verbose=True)
print(Path('eval_outputs/hf_colab_baselines/summary.md').read_text())

## 6. Memory ablation on the HF models (bold step)

Disables memory actions per agent. If memory is load-bearing, removing it should hurt.

In [ ]:
from env.models import QueryMemoryAction, ReflectAction, WaitAction, LinkMemoryAction, ForgetAction

def wrap_no_memory(policy_fn):
    def act(ctx):
        a = policy_fn(ctx)
        if isinstance(a, (QueryMemoryAction, ReflectAction, LinkMemoryAction, ForgetAction)):
            return WaitAction()
        return a
    return act

for m in MODELS:
    print(f'\n=== ablation {m} ===')
    try:
        ablation_policies = {f'hf_no_mem:{m}': wrap_no_memory(build_policy(f'hf:{m}'))}
        ablation_config = EvalConfig(
            policies=ablation_policies,
            scenarios=SCENARIOS,
            seeds=SEEDS,
            out_dir=Path(OUT_DIR) / 'ablation' / m.replace('/', '__'),
        )
        run_eval(ablation_config, verbose=True)
        for key in list(_HF_CACHE.keys()):
            model, tok = _HF_CACHE[key]
            del model; del tok; del _HF_CACHE[key]
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as e:
        print(f'  FAILED: {e}')

## 7. Aggregate + download

Wraps everything into a zip you can pull from the Colab Files panel.

In [ ]:
!python scripts/aggregate_results.py --root eval_outputs --out eval_outputs/COMBINED
print('=== hero table ===')
print(Path('eval_outputs/COMBINED/summary.md').read_text())
!cd {REPO_DIR} && zip -qr /content/hf_colab_results.zip eval_outputs
print('\ndownload: /content/hf_colab_results.zip')

## Interpretation cheatsheet

- `eval_outputs/hf_colab/MODEL_NAME/rollouts.jsonl` â€” one JSON per rollout, for each model Ã— scenario Ã— seed.
- `eval_outputs/hf_colab_baselines/` â€” rule-based reference (sycophant / keyword / memory_aware / contrarian).
- `eval_outputs/hf_colab/ablation/MODEL_NAME/` â€” same model with memory actions disabled.
- `eval_outputs/COMBINED/summary.md` â€” grand aggregated table.

Read the summary: if your HF model has sycophancy_rate > 0, the env caught frontier sycophancy. If its terminal_score > 0, it solved the project. If the ablation delta is positive, memory architecture genuinely helps for that model.